# Phase 7 — Step 72: the noise, power and area budget per block

Phases 1–6 explained *why* each block exists. Phase 7 designs them — and the first design step is to decide how much each block is allowed to cost: jitter, power and area. This notebook splits the proposal's targets (1.4 ps typ / 3.2 ps max with FFNC, −110/−120 dBc/Hz at 300 kHz, 9 mW, 0.08 mm²) into one allocation per block, shows which allocations are critical, and lets you move them with sliders. The same numbers are in `docs/design/step72_budget.md`; the model is `design/budget/step72_budget.py`.

**How to use this notebook.** Every step follows the same loop: **What happens** (the observation) →
**How it happens** (the mechanism and the equations, in Razavi's notation) → **Why it happens** (the physical
reason) → an interactive figure → **Try it** (what to change and what to look for) → **Interview questions**
(click a question to open the answer; try to answer it aloud first).

Notation (Razavi, *RF Microelectronics* ch. 9 and *Design of CMOS PLLs*): $\phi_{in}$, $\phi_{out}$ input and output
phase; $\omega_{out} = \omega_0 + K_{VCO} V_{cont}$; $K_{PD}$ phase-detector gain; $I_p$ charge-pump current;
$R_1, C_1, C_2$ loop filter; $N$ divide ratio; $\omega_n$, $\zeta$ natural frequency and damping;
$H(s) = \Phi_{out}/\Phi_{in}$. Frequencies $f$ in Hz, angular $\omega = 2\pi f$ in rad/s.
Design numbers are the submitted proposal's: $f_{REF} = 50$ MHz, $N = 32$, $f_{out} = 1.6$ GHz, $f_c \approx 2.5$ MHz.

## Contents

- [Step 72 — Phase-noise and power budget per block, and the jitter target](#step-72)

*1 steps, 42 interview questions.*

In [ ]:
# setup - run this cell first (Shift+Enter). Figures are interactive: drag the slider under each plot,
# hover for values, drag to zoom, double-click to reset.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sspll_learn as sl
from sspll_learn import P, show, sweep, bode, kT
np.set_printoptions(precision=4, suppress=True)
print("ready - proposal design point:", f"{P['fref']/1e6:.0f} MHz x {P['N']} = {P['f0']/1e9:.2f} GHz")

<a id='step-72'></a>
## Step 72 — Phase-noise and power budget per block, and the jitter target

*Roadmap step 72: budget table*

### What happens
Every block — crystal, reference receiver, SSPD, charge pump, loop-filter resistor, ring, current source, NCB, LDO — adds its
own small random timing error to the output edges. These errors are independent, so the output jitter is their
**root-sum-square**, each shaped by the path from the block to the output. A budget fixes, *before* any transistor is sized,
how much each block may contribute so the sum meets the specification, and gives each block a pass/fail number in its own
units.

### How it happens
Each source enters at a physical place in the loop; its transfer to the output follows from Phases 3–6:

$$S_{out}(f) = \big[N^2 S_{REF} + S_{PD}\big]\big|H + G_c(1-H)\big|^2 + S_{VCO}\big|(1-G_c)(1-H)\big|^2
 + S_{NCB} + (\omega_0 K_s)^2 S_{v,LDO} + |G_c|^2 S_{alias}$$

$$\sigma_t = \frac{1}{2\pi f_0}\sqrt{\int_{10\,\mathrm{kHz}}^{40\,\mathrm{MHz}} S_{out}(f)\,df},\qquad
\sigma_t^2 = \sum_i \sigma_{t,i}^2 .$$

Each block's physics gives its source in its own units, then converted to dBc/Hz:

| Block | Source | Conversion |
|---|---|---|
| REF (XO + receiver) | floor $L_{REF}$ | $\times N^2$ (+30.1 dB); per-edge jitter $\sigma = \sqrt{L f_{REF}}/(2\pi f_{REF})$ |
| SSPD | $kT/C$ | $L = kT/(2 C_s A^2 f_{REF})$ |
| Pulsed-Gm CP | $4kT\gamma g_m$ during duty $d$ | $L = kT\gamma/(2A^2 g_m d)$, with $\beta = 2A g_m d$ |
| Loop-filter $R_1$ | $4kTR_1$ | $\times |C_1/(C_1+C_2+sR_1C_1C_2)|^2\,(K_{VCO}/\omega)^2$ |
| CCO | $i_n$ (A/√Hz) | $S_\varphi = (K_I i_n)^2/f^2$ |
| NCB / LDO | floor / $v_n$ | 1:1 / $(\omega_0 K_s v_n)^2$ |

### Why it happens this way
The reference is a *time* error, so it is multiplied by $N^2$; the SSPD and CP see the VCO directly, so they are not. The
oscillator's noise is high-passed by the loop $(1-H)$ and again by $(1-G_c)$ with FFNC. The NCB and LDO come *after*
everything, so nothing filters them. Aliasing is the price of FFNC: the SSPD folds VCO noise at $f \pm kf_{REF}$ into the band
and the cancellation path adds it to the output. The relative size of the contributions below follows from these facts.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

rows = {}
for c in ("typ", "pess"):
    for on in (False, True):
        rows[(c, on)] = total(c, on)[0]
print(f"{'block':40s} {'typ off':>8s} {'typ on':>8s} {'pess off':>9s} {'pess on':>8s}   (ps rms, 10 kHz-40 MHz)")
for k in rows[("typ", True)]:
    v = [rows[(c, o)].get(k, 0)*1e12 for c in ("typ", "pess") for o in (False, True)]
    print(f"{k:40s} {v[0]:8.3f} {v[1]:8.3f} {v[2]:9.3f} {v[3]:8.3f}")
print(f"{'spec':40s} {'2.3 typ':>8s} {'1.4 typ':>8s} {'4.7 max':>9s} {'3.2 max':>8s}")

**Figure 72.1 — where the jitter comes from** (typ, FFNC off vs on). Hover for values.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

j0, _, _ = total("typ", False); j1, _, _ = total("typ", True)
names = sorted([k for k in j1 if k != "TOTAL"], key=lambda k: j1[k])
fig = go.Figure()
fig.add_trace(go.Bar(y=names, x=[j0.get(k, 0)*1e12 for k in names], orientation="h", name=f"FFNC off ({j0['TOTAL']*1e12:.2f} ps)", marker_color="#2a78d6"))
fig.add_trace(go.Bar(y=names, x=[j1[k]*1e12 for k in names], orientation="h", name=f"FFNC on ({j1['TOTAL']*1e12:.2f} ps)", marker_color="#eb6834"))
fig.update_layout(barmode="group", height=480, xaxis_title="rms jitter contribution (ps)", title="Jitter budget per block, typ")
show(fig)

**Try it.** Square the bars: the ring's $1.87^2 = 3.5$ ps² is 78% of the FFNC-off variance; the SSPD's $0.014^2$ is 0.004%.
A block at one third of the largest adds only 5% to the total — that is why you do not spend effort on small bars.

**Figure 72.2 — the ring 1/f³ corner and the 300 kHz spec.** The proposal specifies −110 dBc/Hz typ at 300 kHz with FFNC off.
Move the ring's 1/f³ corner: at the simulated 1.1 MHz the spec is missed by ~1.5 dB; the budget allocates ≤ 0.6 MHz.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

def frame(fc3):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, _ = total("typ", on, over={"ring VCO core": (-98.35, fc3)})
        tr.append(dict(x=f, y=L, name=f"FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'}: {j['TOTAL']*1e12:.2f} ps, L(300k) {L[i300]:.1f}")
    tr.append(dict(x=[3e5, 3e5], y=[-110, -120], name="spec typ (off, on)", mode="markers",
                   marker=dict(size=9, color="#59636e", symbol="circle")))
    return tr, f"ring f_c3 = {fc3/1e6:.2f} MHz — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="L(f) at 1.6 GHz (dBc/Hz)", range=[-140, -95])
show(sweep([0.2e6, 0.3e6, 0.4e6, 0.5e6, 0.6e6, 0.8e6, 1.1e6, 1.5e6, 2.2e6], frame, "ring 1/f³ corner",
           fmt="{:.2g} Hz", active=4, layout=lay))

**Figure 72.3 — loop bandwidth as the fallback knob.** If step 74 cannot lower the corner, `icp_trim` raises $f_c$ (it scales
$\beta$). Watch FFNC-off improve while FFNC-on barely moves (the cancellation already removes the VCO noise near $f_c$), and
remember the price: the sample-and-hold delay costs $\approx 180°\,f_c/f_{REF}$ of phase margin.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

def frame(fc):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, _ = total("typ", on, fc=fc)
        tr.append(dict(x=f, y=L, name=f"FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'} {j['TOTAL']*1e12:.2f} ps, L300k {L[i300]:.1f}")
    return tr, f"f_c = {fc/1e6:.1f} MHz (hold-delay PM loss {180*fc/50e6:.0f}°) — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="L(f) (dBc/Hz)", range=[-140, -95])
show(sweep([1.5e6, 2.0e6, 2.5e6, 2.8e6, 3.0e6, 3.5e6, 4.0e6], frame, "loop bandwidth", fmt="{:.2g} Hz", active=2, layout=lay))

**Figure 72.4 — the CCO current noise.** The CCO's thermal floor is fixed by physics: a source of current $I$ with headroom
$V_h$ has $i_n^2 \approx 4kT\cdot 2I/V_h$ → about 7 pA/√Hz at 1.31 mA and 1.2 V. Only its flicker corner is a design choice
(large-area devices, RC-filtered mirror gate). Slide the corner.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

KI = 1.22e12
def frame(fcf):
    tr = []; note = []
    for on, col in ((False, "#2a78d6"), (True, "#eb6834")):
        j, L, S = total("typ", on, over={"CCO current DAC + mirror": (-104.2, fcf)})
        tr.append(dict(x=f, y=10*np.log10(S["CCO current DAC + mirror"]/2), name=f"CCO part, FFNC {'on' if on else 'off'}", line=dict(color=col)))
        note.append(f"{'on' if on else 'off'} total {j['TOTAL']*1e12:.2f} ps (CCO {j['CCO current DAC + mirror']*1e12:.2f})")
    return tr, f"CCO 1/f corner {fcf/1e6:.2f} MHz, i_n = 7.2 pA/√Hz at 1 MHz — " + " | ".join(note)
def lay(fig):
    fig.update_xaxes(type="log", title="offset (Hz)"); fig.update_yaxes(title="CCO contribution at the output (dBc/Hz)", range=[-160, -100])
show(sweep([0.05e6, 0.1e6, 0.2e6, 0.3e6, 0.5e6, 1.1e6, 2.2e6], frame, "CCO flicker corner", fmt="{:.2g} Hz", active=3, layout=lay))

**Figure 72.5 — sensitivity.** Total jitter when one block alone is 3 dB worse than its allocation. Long bars = critical allocations; short bars = slack you can trade for power, area or robustness.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

sens = bud.sensitivity()
base = total("typ", True)[0]["TOTAL"]*1e12
names = sorted(sens, key=lambda k: sens[k][1])
fig = go.Figure(go.Bar(y=names, x=[sens[k][1] - base for k in names], orientation="h", marker_color="#eb6834",
                       name="FFNC on", hovertemplate="%{y}: +%{x:.3f} ps<extra></extra>"))
fig.update_layout(height=430, xaxis_title=f"increase of total typ jitter (ps) over {base:.2f} ps", title="Block +3 dB → total jitter (FFNC on)")
show(fig)

### Power and area

Blocks on the 1.8 V LDO rail draw their current from VDDA = 3.3 V, so every mA there costs **3.3 mW**; the ring is fed from
VDDA through the CCO (4.86 mW). Logic that can run from VDDD costs 1.8 mW/mA — that is why dividers and FLL logic live there.

In [ ]:
import sys; sys.path.insert(0, "../models"); sys.path.insert(0, "../design/budget")
import pllmodel as pm, step72_budget as bud
f = pm.freq_axis(n=700)
def total(corner, on, **kw):
    ff, S = bud.spectra(corner, on, f=f, **kw)
    j, L = bud.jitter(S, ff, 1.6e9)
    return j, L, S
i300 = int(abs(f - 3e5).argmin())

tot = {"on": 0, "off": 0, "on_w": 0, "off_w": 0}
print(f"{'block':46s} {'supply':6s} {'I (mA)':>7s} {'P typ':>7s} {'P worst':>8s}")
for name, sup, i, p, pw, ncb in bud.power_table():
    print(f"{name:46s} {sup:6s} {i:7.3f} {p:7.2f} {pw:8.2f}")
    tot["on"] += p; tot["on_w"] += pw
    if not ncb: tot["off"] += p; tot["off_w"] += pw
print(f"TOTAL FFNC on {tot['on']:.2f} mW (worst {tot['on_w']:.2f}; spec 9.0 typ / 15 max)")
print(f"TOTAL FFNC off {tot['off']:.2f} mW (worst {tot['off_w']:.2f}; spec 7.5 typ / 13 max)")
j = total("typ", True)[0]["TOTAL"]
print(f"FoM = {10*np.log10(j**2*tot['on']):.1f} dB (proposal -227.5)")
print("\narea:", ", ".join(f"{n} {a}" for n, a in bud.AREA), f"\n-> {sum(a for _, a in bud.AREA)} um^2 (spec 80,000 typ)")

### Layout convenience, decided now

The budget tells you which nodes are noise-critical, so the schematic can already be drawn the way it will be laid out:

- **Ring (dominant):** 5 identical cells, U-shaped placement so the wrap-around wire matches the others, dummy loads on unused
  taps, integer fingers per device so each cell is one abutting row.
- **CCO (second):** 1:8 mirror as 9 unit devices in a common-centroid row with end dummies; binary DAC from unit cells;
  RC-filtered gate (long high-resistance poly + MOS cap under the array); 3.3 V devices in their own guard ring.
- **Reference receiver (×N²):** short, shielded route from the pad; own VDDA branch; far from the ring.
- **SSPD, CP:** fully differential, mirror-symmetric, unit MIM tiles with a dummy ring, ABBA input pair, non-minimum L.
- **Loop filter:** R1 as equal legs of wide poly; C1 MOS cap with MIM stacked above (~9.5 fF/µm², V_cont > 0.6 V); C2 linear MIM; tiles ≤ 30×30 µm.
- **NCB / LDO:** NCB at the output buffer with its own LDO decap; LDO pass device as unit fingers along the supply edge.
- **Everywhere:** few distinct unit devices; supply domains as bands (VDDA / LDO 1.8 V / VDDD); level shifters only at band edges.

### Interview questions — step 72 (42)

<details><summary><b>Q1.</b> What is a noise budget, and why do it before designing any transistor?</summary>

A table that splits the system target (here 1.4 ps typ / 3.2 ps max rms jitter, −110/−120 dBc/Hz at 300 kHz) into one allocation per block, each in that block's own units (pA/√Hz, nV/√Hz, fs per edge). It turns one system number into pass/fail tests a single block can be designed and simulated against, and it shows *early* which block dominates — so the design effort (and power and area) goes there and not to blocks that cannot matter.

</details>

<details><summary><b>Q2.</b> Why do jitter contributions add in power (rms²) and not linearly?</summary>

The sources are physically independent (different devices), so they are uncorrelated and their variances add: σ² = Σσᵢ². Consequence: a block at half the jitter of the largest one adds only 12% to the total; a block at 1/3 adds 5%. Fixing small contributors is wasted effort.

</details>

<details><summary><b>Q3.</b> Write the output phase-noise PSD of the SSPLL with FFNC in terms of the block sources.</summary>

S_out(f) = [S_REF·N² + S_PD,eq]·|H + G_c(1−H)|² + S_VCO,eq·|(1−G_c)(1−H)|² + S_NCB + S_alias·|G_c|², where H is the closed-loop transfer, G_c the cancellation path (SSPD gain × NCB gain, with sinc hold, half-period delay and pole) and the VCO-like sources (ring, CCO, loop-filter R) all enter at the VCO. Jitter = √(∫S_out df)/(2πf_out), here 10 kHz–40 MHz.

</details>

<details><summary><b>Q4.</b> Why is the reference noise multiplied by N² but the SSPD noise is not?</summary>

The reference sets the *time* of each edge; a time error Δt at f_REF becomes a phase error 2πf_out·Δt at the output, i.e. N times the reference phase error → N² in power. The SSPD compares the VCO directly (no divider), so its voltage noise divided by its gain is already a VCO-phase error: no N. That is the whole point of sub-sampling (−130 vs −110.9 dBc/Hz detector floor at the output, step 8).

</details>

<details><summary><b>Q5.</b> Why does FFNC make the reference contribution *worse* (0.20 → 0.31 ps)?</summary>

With FFNC on, the in-band transfer becomes |H + G_c(1−H)|²: above the loop bandwidth the cancellation path also passes the SSPD's view of the reference, so reference (and SSPD/CP) noise reaches the output up to ≈f_REF/8 instead of f_c. FFNC trades VCO noise for reference noise; it wins because the VCO term is 10× larger.

</details>

<details><summary><b>Q6.</b> Which block dominates the budget, and what does that tell you?</summary>

The ring VCO core (1.87 ps off / 1.13 ps on typ), then the CCO (0.90 / 0.56). Everything else together is < 0.5 ps. So power, area and design time go to the oscillator and its current source; the SSPD, CP and NCB can be designed for linearity, spurs and robustness rather than noise.

</details>

<details><summary><b>Q7.</b> The SSPD kT/C noise contributes only 22 fs. Why not shrink the sampling capacitor to save power?</summary>

Because the cap also sets the charge-sharing/kick-back spur and the hold droop; and the SSPD noise is in-band where FFNC passes it. 100 fF keeps kT/C at −147.8 dBc/Hz (17 dB below the CP allocation), so it is not the limit; going to 25 fF costs +6 dB noise but buys little power (the ring buffers that drive it set the power). The budget says: free to choose the cap for spur and matching, as long as Cs ≥ ~50 fF per side.

</details>

<details><summary><b>Q8.</b> Derive the SSPD kT/C phase-noise floor.</summary>

Each side samples kT/C; the differential sample has v² = 2kT/C. The differential slope is 2A per radian of VCO phase, so one sample carries φ² = (2kT/C)/(2A)² = kT/(2CA²). Samples are independent, so this variance is white over 0…f_REF/2: S_φ = φ²/(f_REF/2) = kT/(C·A²·f_REF), and L = S_φ/2 = kT/(2·C·A²·f_REF) = −147.8 dBc/Hz for C = 100 fF, A = 0.5 V, f_REF = 50 MHz.

</details>

<details><summary><b>Q9.</b> How is the pulsed-Gm charge-pump noise budgeted, and why does the duty cycle matter?</summary>

The transconductor's output noise current 4kTγg_m only reaches the loop filter during the pulse (duty d ≈ 1%, 200 ps of 20 ns). The useful signal current is β = 2A·g_m·d (A per rad). Referred to VCO phase: S_φ = 4kTγg_m·d/β² → L ≈ kTγ/(2A²g_m d) ≈ −140.8 dBc/Hz white, plus a 1/f corner allocated ≤ 300 kHz. Shorter pulses lower the signal more than the noise at equal g_m, so d is set by β and the spur, not by noise.

</details>

<details><summary><b>Q10.</b> Why is the loop-filter resistor a visible contributor (0.28 ps) and what are the knobs?</summary>

Its 4kTR voltage noise (12.6 nV/√Hz for 9.61 kΩ) is filtered by C1/(C1+C2+sRC1C2) and integrated by K_VCO/s, so it behaves like VCO noise (high-passed by 1−H, cancelled by FFNC). Knobs: larger β (more CP current → smaller R, larger C, more area), smaller K_VCO (needs a wider coarse range from the CCO DAC), or a dual-path filter. We accept it: 0.28 ps adds only ~1% to a 2.1 ps total.

</details>

<details><summary><b>Q11.</b> Why must the ring 1/f³ corner be halved (1.1 → 0.6 MHz) in this budget?</summary>

With the simulated 1.1 MHz corner the proposal's own model gives L(300 kHz) = −108.1 dBc/Hz FFNC-off typ, 1.9 dB short of the −110 typ in the specification. At 300 kHz the ring is in its 1/f³ region, so L ∝ (1 + f_c3/f); halving f_c3 plus a lower-flicker CCO brings it to −110.2. Halving needs ~2× gate area in the ring transistors that upconvert flicker, or better rise/fall symmetry (smaller DC term of the ISF, c₀).

</details>

<details><summary><b>Q12.</b> What determines the 1/f³ corner of a ring oscillator?</summary>

Hajimiri–Lee: f_1/f³ = f_1/f(device) · (c₀/c_rms)². c₀ is the DC value of the ISF (zero for a perfectly symmetric waveform), c_rms its rms. So the corner falls with device area (lower device flicker corner) and with symmetry of rise and fall edges (pMOS/nMOS strength balance, equal loading).

</details>

<details><summary><b>Q13.</b> Why is the CCO noise allocated in pA/√Hz, and how do you convert it?</summary>

The CCO is a current source: what it delivers to the ring is a noisy current. Frequency noise = K_I·i_n (K_I = ∂f/∂I = 1.22×10¹² Hz/A), phase noise S_φ = (K_I i_n)²/f², L = S_φ/2. L(1 MHz) = −104.2 dBc/Hz ↔ i_n = 7.2 pA/√Hz.

</details>

<details><summary><b>Q14.</b> Why can't the CCO noise be made arbitrarily small?</summary>

A MOS current source of current I with overdrive (headroom) V_ov has i_n² ≈ 4kTγ·2I/V_ov. At 1.31 mA and ~1.2 V of headroom left by VDDA_min 2.97 V minus the ~1.6–1.8 V ring node, the thermal floor is ≈6–7 pA/√Hz. Degeneration gives the same limit (4kT/R_s with R_s = V_h/I). Only more headroom or less current helps — the budget therefore allocates the thermal floor and attacks only the flicker part.

</details>

<details><summary><b>Q15.</b> Why does a filtered mirror gate help the CCO noise?</summary>

In a 1:8 mirror the reference-side noise (DAC unit cells, IBIAS mirror, diode device) is multiplied by 8 at the output. An RC low-pass on the shared gate (corner ≪ 10 kHz, e.g. a long poly resistor and a MOS cap) removes it inside the integration band, so only the output devices' own noise remains. It costs area, not power.

</details>

<details><summary><b>Q16.</b> Why is the NCB intrinsic noise budget so small (31 fs) yet still important?</summary>

It is added *after* the loop and the cancellation: nothing filters it, so it appears 1:1 at the output with FFNC on, all the way to 40 MHz. −150 dBc/Hz white with a 1 MHz flicker corner gives 31 fs. If the delay line were a long chain of starved inverters, its noise would grow with its delay and could reach hundreds of fs — so the NCB is kept short (2 cells, small delay range).

</details>

<details><summary><b>Q17.</b> How does LDO noise get into the output, and how is that budgeted?</summary>

The NCB delay depends on its supply (≈104 ps/V for 4 inverter edges at ~26 ps/V each): a supply noise density v_n becomes a timing noise K_s·v_n, i.e. phase 2πf_out·K_s·v_n. For ≤ 26 fs contribution the LDO output noise must be ≤ 40 nV/√Hz (10 kHz flicker corner). The ring itself is fed by the high-impedance CCO, not the LDO, so LDO noise does not modulate the oscillator directly.

</details>

<details><summary><b>Q18.</b> What is the aliasing term in the FFNC budget?</summary>

The SSPD samples at f_REF, so VCO noise at f ± k·f_REF folds into the 0…f_REF/2 band (Thijssen: S_s(f) = f_REF Σ S_φ(f − k f_REF)). The cancellation path then *adds* this folded noise to the output (it is not correlated with the output's own noise at f). It costs 0.19 ps typ — the main reason FFNC cannot cancel everything.

</details>

<details><summary><b>Q19.</b> Why is the jitter integrated from 10 kHz to 40 MHz?</summary>

The proposal's specification band. The lower edge excludes slow wander a downstream system tracks; the upper edge is the Nyquist range of a 100 MHz-class sampling clock divided output and where the ring's −20 dB/dec skirt has fallen well below the in-band level. A budget must use the same band as the spec.

</details>

<details><summary><b>Q20.</b> How do you check a budget against the reference range 45–55 MHz?</summary>

Re-run with f_REF = 45 and 55 MHz: f_out scales, the ring's L at 1 MHz scales ~ f_out (same power per edge), the aliasing and the cancellation bandwidth move with f_REF. Typ jitter goes 2.23/1.51 ps at 45 MHz to 2.01/1.23 ps at 55 MHz (off/on) — all inside the maxima.

</details>

<details><summary><b>Q21.</b> What does the 'pessimistic' column represent?</summary>

A coherent worst plausible set for the noise numbers: ring +1 dB and 2× 1/f³ corner, CCO +3 dB and 2× corner, reference receiver +4 dB with a 100 kHz corner, SSPD amplitude 0.35 V, γ = 1.5, NCB −146 dBc/Hz, LDO 80 nV/√Hz and a 30% FFNC gain error. It is not a SPICE corner; it is the budget's stress case that must still meet the *maximum* spec.

</details>

<details><summary><b>Q22.</b> The FFNC-on L(300 kHz) is −119.9 typ against a −120 typ spec. Is that a fail?</summary>

It is a typ number at the limit, not a guaranteed limit (the max is −110, with 4.5 dB margin pess). At 300 kHz with FFNC on the floor is set by the reference ×N²: crystal −155 and receiver −155 → −121.2 dBc/Hz together, before any on-chip noise. Only a cleaner receiver (−157 for ~+0.1 mW) or a better board crystal moves it. This is recorded as a finding.

</details>

<details><summary><b>Q23.</b> How is the power budget built and why does the LDO domain cost more?</summary>

Every block gets a current on a named supply; power = I × supply voltage at the slot. Blocks on the 1.8 V LDO rail draw their current from VDDA = 3.3 V, so each mA costs 3.3 mW, not 1.8 mW: the LDO burns the 1.5 V difference. That is why logic that can live on VDDD (1.8 V, 1.8 mW/mA) is put there.

</details>

<details><summary><b>Q24.</b> Why is the ring fed from VDDA through a CCO instead of from the LDO?</summary>

A current-fed ring has low supply sensitivity (the CCO's output impedance isolates it) and the current DAC also gives the coarse tuning. The price: the ring burns its current at 3.3 V (4.86 mW). Running it from the LDO would cost the same power from VDDA and add the LDO noise directly to the oscillator.

</details>

<details><summary><b>Q25.</b> What is the worst-case power and how is it estimated?</summary>

Ring at 1.76 GHz, ss, 110 °C, VDDA 3.63 V: 7.46 mW (from the proposal's simulated basis). Other blocks: typ current × 1.3 at the maximum supply. Total 13.5 mW FFNC on / 11.4 mW off against 15/13 mW maximum.

</details>

<details><summary><b>Q26.</b> What is the jitter figure of merit and what does this budget give?</summary>

FoM = 10·log₁₀[(σ/1 s)²·(P/1 mW)]. σ = 1.36 ps, P = 9.08 mW → −227.8 dB (proposal −227.5). Note the FoM rewards power and jitter equally in dB: halving the jitter (−6 dB) is worth 4× the power.

</details>

<details><summary><b>Q27.</b> Why does the area table now include the LDO?</summary>

Decision D4 (step 8) added an on-chip LDO for the clean 1.8 V rail; the proposal's area basis listed every block except it. ~4000 µm² (pass device for ~1.2 mA, error amp, compensation and output cap partly as MOS cap). Total 57,500 µm² — still below the 0.08 mm² typ.

</details>

<details><summary><b>Q28.</b> How would you decide between spending 1 mW on a cleaner VCO and 1 mW on a better NCB?</summary>

Use the sensitivity of the total to each block: ring +3 dB moves the typ total from 1.36 to 1.77 ps; NCB +3 dB changes nothing visible. The marginal improvement per mW is far larger at the VCO. A budget makes that comparison numeric.

</details>

<details><summary><b>Q29.</b> What is the 'sensitivity' table and how is it used during design?</summary>

For each block, the total jitter when that block alone is 3 dB worse than allocated. It shows which allocations are critical (ring, CCO, reference) and which have slack (SSPD, CP, NCB, LDO). Slack can be traded for power, area or robustness during the block designs.

</details>

<details><summary><b>Q30.</b> How will each allocation be verified in Phase 8?</summary>

Each block's testbench measures its allocation in its own units: SSPD/CP with .noise and PSS-equivalent transient noise (tb_sspd_char, tb_sscp_*), ring free-running phase noise by transient noise and ISF (tb_vco_*), CCO output current noise by .noise (tb_cco_ac), LDO output noise and PSRR (tb_ldo_ac), NCB delay noise and supply sensitivity (tb_ncb_*). The results replace the allocations in step72_budget.py, which recomputes the total.

</details>

<details><summary><b>Q31.</b> Why keep the budget in a script instead of a spreadsheet?</summary>

Because the transfer functions are frequency-dependent and complex (H, 1−H, G_c, aliasing) — a spreadsheet of single numbers cannot integrate them. A script also re-runs automatically when a simulated curve replaces an allocation, and it is version-controlled next to the schematics.

</details>

<details><summary><b>Q32.</b> What layout decisions follow directly from the noise budget?</summary>

(1) The ring and its CCO get the cleanest, symmetric placement (dominant noise); (2) the CCO output mirror is large-area, common-centroid, with a filtered gate; (3) the reference receiver path is short and shielded (it is ×N²); (4) the SSPD caps are placed symmetrically with dummies (spur, not noise); (5) the NCB is next to the output with its own LDO decap; (6) the loop-filter C1 stacks MIM over MOS to fit the area.

</details>

<details><summary><b>Q33.</b> Why is the loop-filter C1 a MOS capacitor with MIM on top, and what is the catch?</summary>

C1 = 18.2 pF would need ~9100 µm² as MIM alone (~2 fF/µm²), more than the CP+LF area allocation. A thin-oxide MOS cap (7.5 fF/µm² at 0.9 V) with a MIM cap stacked above gives ~9.5 fF/µm² → ~1,900 µm² (~11.5 fF/µm² with both MIM layers). Catch: a MOS cap is non-linear and must be kept in strong inversion/accumulation across the tuning range — acceptable for C1 (the large zero cap), not for C2.

</details>

<details><summary><b>Q34.</b> What is the effect of a loop-gain (K_VCO or β) error on the budget?</summary>

f_c moves (f_c ∝ βK_VCO): a lower f_c lets more VCO noise through and raises FFNC-off jitter; a higher f_c helps here (2.8 MHz closes the 300 kHz spec alone) but costs phase margin through the sample-and-hold delay (≈ 180°·f_c/f_REF). icp_trim[1:0] is the knob to recentre f_c after silicon or after the K_VCO of step 74 is known.

</details>

<details><summary><b>Q35.</b> Why 2× margin between pessimistic budget and the max spec?</summary>

The budget's models (ISF-based ring noise, γ, flicker coefficients) carry several-dB uncertainty, and layout parasitics add noise and power. The proposal says maximum values keep about twice the pre-layout margin; here pess 2.71/1.73 ps vs 4.7/3.2 ps max is 1.7–1.8×.

</details>

<details><summary><b>Q36.</b> Explain 'L = S_φ/2' and why the code uses 2·10^(L/10).</summary>

L(f) is defined as single-sideband noise relative to the carrier; for small phase noise L(f) = S_φ(f)/2 where S_φ is the one-sided phase PSD. Integrating jitter needs S_φ = 2·10^(L/10) rad²/Hz, then σ_t = √∫S_φ df / (2πf_0).

</details>

<details><summary><b>Q37.</b> How does a flicker (1/f) noise source in the CP or SSPD show up at the output?</summary>

Below the loop bandwidth H ≈ 1, so a 1/f input source appears as 1/f at the output — it raises L(f) at low offsets (10–100 kHz). That is why the CP/SSPD allocations include a 300 kHz flicker corner and why their devices are not minimum length.

</details>

<details><summary><b>Q38.</b> What would happen to the budget if the reference were 25 MHz with N = 64?</summary>

The reference term grows by 20·log(2) = 6 dB (N²), the SSPD term grows because A²·f_REF halves (+3 dB), the cancellation bandwidth halves (f_REF/8) and aliasing worsens — the 50 MHz reference is a deliberate budget choice.

</details>

<details><summary><b>Q39.</b> Which allocations are measured on silicon, and how?</summary>

Total jitter and L(f) on CLK_OUT (/16 + 24.1 dB), FFNC on vs off on the same chip, supply sensitivity with the VTEST tone, and power per domain from slot currents with blocks enabled in turn. Individual block noise is not directly observable — the budget lets you attribute the measured total.

</details>

<details><summary><b>Q40.</b> A colleague proposes a 3-stage ring to save power. What does the budget say?</summary>

The step-6 study: 3 × 1.0 at 0.61 mA gives −87.4 dBc/Hz vs −94.2 for 5 × 1.5 — 6.8 dB worse at the dominant block; jitter would roughly double. Saving ~2 mW does not pay; the 5-stage verdict stands.

</details>

<details><summary><b>Q41.</b> Why is the FLL charge pump nearly absent from the noise budget?</summary>

It has a dead zone: in lock the phase error sits inside it and the FLL CP is off — no noise current reaches the loop filter. Its leakage and switch charge injection matter for spurs (step 73), not for random jitter.

</details>

<details><summary><b>Q42.</b> What are the three numbers every block designer must take away from step 72?</summary>

Its noise allocation (in its own units), its supply current on a named supply, and its area. Plus its layout rules from the budget: which nodes are noise-critical, which must be symmetric and which supply domain it lives in.

</details>